# Visium HD Spatial Transcriptomics Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenpastor/spatial_transcriptomics_resources/blob/main/notebook/visium_hd_crc_p1_tutorial.ipynb)



This tutorial demonstrates a partial Visium HD spatial transcriptomics workflow using
the **10x Genomics Human Colorectal Cancer (CRC), Patient 1** dataset from the
[Nature Genetics 2025 publication](https://www.nature.com/articles/s41588-025-02193-3)

**Why this dataset?**
- **Has ground-truth cell type annotations** from the 10x Genomics
  [HumanColonCancer_VisiumHD](https://github.com/10XGenomics/HumanColonCancer_VisiumHD)
  repo; 38 deconvolved cell types + unsupervised clusters + tumor periphery labels
- **Hetergeneous enough biology**: tumor, stroma, immune infiltrate (macrophage subtypes, T cells, B cells),
  intestinal epithelium (goblet cells, enterocytes, tuft cells)
- **Decent QC variation**: expect to filter 5-15% of bins (unlike some of the ultra-sparse toy datasets)

**What you will learn:**
1. Load pre-processed Visium HD data (8 µm resolution, 50K bins)
2. Quality control with real filtering (per-bin metrics + spatial outlier detection)
3. Compare your marker-based annotations against published ground-truth labels
4. Spatial neighborhood analysis, spatially variable genes, cell-cell communication

**What is not yet included**
- **Segmentation using 2 µm resolution**: harder to include that here but later I will provide a pre-processed dataset demo

**Requirements**: Google Colab (free tier is sufficient) and all data is downloaded automatically

---


## Section 0. Installation and Data Retrieval

In [ ]:
# Package installation + data download
import os, sys, subprocess
from pathlib import Path

# Install packages
print("Installing packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "scanpy>=1.10", "squidpy>=1.4", "anndata>=0.10",
    "numpy>=1.24", "pandas>=2.0", "scipy>=1.11",
    "matplotlib>=3.7", "seaborn>=0.13",
    "scikit-image>=0.21", "Pillow>=10",
    "igraph", "leidenalg",
    "tqdm>=4.65", "requests>=2.31", "h5py>=3.9", "pyarrow>=13",
])
print("Done.")

# Download precomputed data from Figshare (per-file via API, no zip)
FIGSHARE_ARTICLE_ID = "31937262"
FIGSHARE_API = f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}/files"

DATA_DIR = Path("/content/data")
PRECOMPUTED_DIR = DATA_DIR / "precomputed"
SPATIAL_DIR = PRECOMPUTED_DIR / "spatial"
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)
SPATIAL_DIR.mkdir(parents=True, exist_ok=True)

RAW_H5AD = PRECOMPUTED_DIR / "crc_p1_raw_50k.h5ad"
ANNOT_H5AD = PRECOMPUTED_DIR / "crc_p1_annotated_50k.h5ad"

SPATIAL_FILENAMES = {
    "tissue_hires_image.png",
    "tissue_lowres_image.png",
    "scalefactors_json.json",
}

def _target_path(name):
    return SPATIAL_DIR / name if name in SPATIAL_FILENAMES else PRECOMPUTED_DIR / name

def download_file(url, dest, desc="Downloading"):
    """Stream a single file to disk with a progress bar and atomic rename."""
    import requests
    from tqdm.auto import tqdm
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=(60, None)) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=desc) as pbar:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    tmp.replace(dest)
    print(f"Saved: {dest.relative_to(DATA_DIR)} ({dest.stat().st_size / 1e6:.2f} MB)")

def fetch_figshare_files():
    import requests
    resp = requests.get(FIGSHARE_API, timeout=60)
    resp.raise_for_status()
    return resp.json()

needed = [
    RAW_H5AD,
    ANNOT_H5AD,
    SPATIAL_DIR / "tissue_hires_image.png",
    SPATIAL_DIR / "tissue_lowres_image.png",
    SPATIAL_DIR / "scalefactors_json.json",
]

if not all(p.exists() for p in needed):
    print("Downloading precomputed data from Figshare (per file)...")
    for entry in fetch_figshare_files():
        name = entry["name"]
        dest = _target_path(name)
        expected_size = entry.get("size")
        if dest.exists() and expected_size is not None and dest.stat().st_size == expected_size:
            print(f"Skipping (already present): {dest.relative_to(DATA_DIR)}")
            continue
        download_file(entry["download_url"], dest, desc=name)
    print("Download complete.")
else:
    print("Precomputed data already downloaded.")

# Clone repo for utils.py
REPO_URL = "https://github.com/stevenpastor/spatial_transcriptomics_resources.git"
REPO_DIR = Path("/content/spatial_tutorial")
if not REPO_DIR.exists():
    print("Cloning tutorial repo (for utility functions)...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
SCRIPTS_DIR = REPO_DIR / "scripts"
print("Ready.")

In [ ]:
import sys, os, time, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False, fontsize=12)

# Paths
DATA_DIR = Path("/content/data")
PRECOMPUTED_DIR = DATA_DIR / "precomputed"
FIG_DIR = Path("/content/figures")
FIG_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(SCRIPTS_DIR))
from utils import (
    load_visium_hd_binned,
    compute_qc_metrics,
    spatial_qc_plots,
    spatial_outlier_detection,
)

print(f"scanpy {sc.__version__}, squidpy {sq.__version__}")


## Section 1: Data Loading

We load pre-processed 8 µm binned data (50K bins) from the CRC P1 dataset.
The precomputed files include ground-truth cell type annotations and an embedded H&E image.


In [ ]:
# Precomputed data is already downloaded above
import os
for f in sorted(PRECOMPUTED_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name:40s}  {size_mb:6.1f} MB" if f.is_file() else f"  {f.name}/")


In [ ]:
%%time
# Load 8 µm binned data
adata_8um = sc.read_h5ad(PRECOMPUTED_DIR / "crc_p1_raw_50k.h5ad")
print(f"Loaded: {adata_8um.shape[0]:,} bins x {adata_8um.shape[1]:,} genes")

print(f"\nSpatial range X: [{adata_8um.obsm['spatial'][:,0].min():.0f}, {adata_8um.obsm['spatial'][:,0].max():.0f}]")
print(f"Spatial range Y: [{adata_8um.obsm['spatial'][:,1].min():.0f}, {adata_8um.obsm['spatial'][:,1].max():.0f}]")
if sparse.issparse(adata_8um.X):
    print(f"Total UMIs: {adata_8um.X.sum():,.0f}")
    print(f"Median UMIs/bin: {np.median(adata_8um.X.sum(axis=1).A1):.0f}")
    print(f"Median genes/bin: {np.median((adata_8um.X > 0).sum(axis=1).A1):.0f}")
else:
    X = adata_8um.X
    print(f"Total UMIs: {X.sum():,.0f}")
    print(f"Median UMIs/bin: {np.median(X.sum(axis=1)):.0f}")
    print(f"Median genes/bin: {np.median((X > 0).sum(axis=1)):.0f}")


In [ ]:
# Load and attach ground-truth annotations

if "UnsupervisedL1" in adata_8um.obs.columns:
    print("Ground-truth annotations already present (from precomputed data).")
    print(f"UnsupervisedL1 coverage: {adata_8um.obs['UnsupervisedL1'].notna().sum():,} / {adata_8um.shape[0]:,}")
    print()
    print("UnsupervisedL1 distribution:")
    print(adata_8um.obs["UnsupervisedL1"].value_counts().to_string())
else:
    print("Loading ground-truth metadata...")
    if not METADATA_PATH.exists():
        download_file(METADATA_URL, METADATA_PATH, desc="Metadata")

    meta = pd.read_parquet(METADATA_PATH)
    meta["tissue"] = meta["tissue"].astype(str)
    meta = meta[meta["tissue"] == "1"].copy()
    meta = meta.set_index("barcode")

    common = adata_8um.obs_names.intersection(meta.index)
    print(f"Matched {len(common):,} / {adata_8um.shape[0]:,} bins to ground-truth metadata")

    for col in ["DeconvolutionClass", "DeconvolutionLabel1", "DeconvolutionLabel2",
                "UnsupervisedL1", "UnsupervisedL2", "Periphery"]:
        if col in meta.columns:
            series = meta.reindex(adata_8um.obs_names)[col]
            if hasattr(series, "cat"):
                series = series.astype(object)
            adata_8um.obs[col] = series.values

    print(f"\nUnsupervisedL1 coverage: {adata_8um.obs['UnsupervisedL1'].notna().sum():,} / {adata_8um.shape[0]:,}")
    print(f"DeconvolutionLabel1 coverage: {adata_8um.obs['DeconvolutionLabel1'].notna().sum():,}")
    print()
    print("UnsupervisedL1 distribution:")
    print(adata_8um.obs["UnsupervisedL1"].value_counts().to_string())


In [ ]:
# 16 µm comparison skipped; precomputed data uses 8 µm only
print("16 µm comparison not available in precomputed mode.")


## Section 2: Image Verification

Verify that the 8 µm bin positions correctly overlay the H&E tissue image.
The spatial coordinates must be scaled by the `tissue_hires_scalef` factor.

**Note:** The right panel shows the *entire* H&E image (not a zoomed-in crop or tissue subregion) — the same full image as the left panel, with bin positions overlaid on top. The bins look sparse because the precomputed `crc_p1_raw_50k.h5ad` is a random subsample of ~50K bins drawn from the full ~500K+ 8 µm bins (done to keep the file under GitHub's 100 MB limit; see `scripts/generate_precomputed.py`). Every bin is drawn from the full tissue, just thinned by roughly 10×.

In [ ]:
%%time
# Load H&E and overlay bin positions
import json as _json

# Image is embedded in the h5ad by generate_precomputed.py
if "spatial_image" in adata_8um.uns:
    he_image = adata_8um.uns["spatial_image"]
    scalefactors = adata_8um.uns.get("scalefactors", {})
    sf = scalefactors.get("tissue_hires_scalef", scalefactors.get("tissue_lowres_scalef", 1.0))
    print(f"Image from adata.uns, shape={he_image.shape}, sf={sf}")
else:
    # Fallback: check spatial/ directory
    from skimage import io as skio
    spatial_dir = PRECOMPUTED_DIR / "spatial"
    he_image = None
    for name in ["tissue_hires_image.png", "tissue_lowres_image.png"]:
        candidate = spatial_dir / name
        if candidate.exists():
            he_image = skio.imread(str(candidate))
            with open(spatial_dir / "scalefactors_json.json") as f:
                scalefactors = _json.load(f)
            sf = scalefactors.get("tissue_hires_scalef", scalefactors.get("tissue_lowres_scalef", 1.0))
            break

if he_image is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    axes[0].imshow(he_image)
    axes[0].set_title("H&E Image")
    axes[0].axis("off")

    coords = adata_8um.obsm["spatial"]
    axes[1].imshow(he_image)
    axes[1].scatter(
        coords[:, 0] * sf, coords[:, 1] * sf,
        s=0.1, c="red", alpha=0.2, rasterized=True,
    )
    axes[1].set_title("8 µm bins overlaid on H&E")
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_image_verification.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Bins plotted: {coords.shape[0]:,}")
else:
    print("H&E image not found. Plotting spatial coordinates only.")
    fig, ax = plt.subplots(figsize=(8, 7))
    coords = adata_8um.obsm["spatial"]
    total_counts = np.asarray(adata_8um.X.sum(axis=1)).flatten() if sparse.issparse(adata_8um.X) else adata_8um.X.sum(axis=1)
    sc_plot = ax.scatter(coords[:, 0], coords[:, 1], s=0.1,
                         c=total_counts,
                         cmap="viridis", edgecolors="none", rasterized=True)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    ax.set_title("Bin positions (colored by UMI count)")
    plt.colorbar(sc_plot, ax=ax, shrink=0.6)
    plt.tight_layout()
    plt.show()


## Section 3: Quality Control — Part A: Per-Bin Metrics

| Metric | What it measures | High values suggest |
|--------|-----------------|-------------------|
| `total_counts` | Total UMIs per bin | Dense tissue / doublets |
| `n_genes_by_counts` | Genes detected per bin | Transcriptomic complexity |
| `pct_counts_mt` | % mitochondrial UMIs | Dying/stressed cells, tissue damage |
| `pct_counts_ribo` | % ribosomal UMIs | Active translation |
| `complexity` | log(genes)/log(counts) | Transcriptomic diversity |

This CRC tissue has genuine quality variation: tissue folds, necrotic regions,
and edge artifacts. Expect to filter 5–15% of bins.

In [ ]:
%%time
# Compute QC metrics
adata_8um = compute_qc_metrics(adata_8um)

print("QC metric summary:")
for col in ["total_counts", "n_genes_by_counts", "pct_counts_mt", "pct_counts_ribo", "complexity"]:
    vals = adata_8um.obs[col]
    print(f"  {col:25s}  median={vals.median():8.1f}  mean={vals.mean():8.1f}  "
          f"[{vals.min():.1f}, {vals.max():.1f}]")

In [ ]:
# Violin plots
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, metric in zip(axes, ["total_counts", "n_genes_by_counts",
                              "pct_counts_mt", "pct_counts_ribo", "complexity"]):
    parts = ax.violinplot(adata_8um.obs[metric].dropna().values, showmedians=True, showextrema=False)
    for pc in parts["bodies"]:
        pc.set_facecolor("steelblue"); pc.set_alpha(0.7)
    ax.set_title(metric.replace("_", "\n"), fontsize=10)
    ax.set_xticks([])
plt.suptitle("QC Distributions (8 µm bins)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_violins.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Scatter: n_genes vs total_counts colored by pct_mt
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    adata_8um.obs["total_counts"], adata_8um.obs["n_genes_by_counts"],
    c=adata_8um.obs["pct_counts_mt"], s=0.5, cmap="RdYlBu_r",
    edgecolors="none", rasterized=True,
    vmin=0, vmax=adata_8um.obs["pct_counts_mt"].quantile(0.95),
)
ax.set_xlabel("Total UMI counts"); ax.set_ylabel("Genes detected")
ax.set_title("Gene complexity colored by mitochondrial %")
plt.colorbar(scatter, ax=ax, label="pct_counts_mt")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Histograms with data-driven threshold lines
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

min_counts = max(adata_8um.obs["total_counts"].quantile(0.01), 10)
max_counts = adata_8um.obs["total_counts"].quantile(0.995)
min_genes = max(adata_8um.obs["n_genes_by_counts"].quantile(0.01), 5)
max_mt = min(adata_8um.obs["pct_counts_mt"].quantile(0.95), 25)

thresholds = [
    ("total_counts", min_counts, max_counts, "UMI Counts"),
    ("n_genes_by_counts", min_genes, None, "Genes Detected"),
    ("pct_counts_mt", None, max_mt, "Mitochondrial %"),
    ("pct_counts_ribo", None, None, "Ribosomal %"),
    ("complexity", None, None, "Complexity"),
]

for ax, (metric, lo, hi, label) in zip(axes.flat, thresholds):
    ax.hist(adata_8um.obs[metric].dropna(), bins=100, color="steelblue", alpha=0.7, edgecolor="none")
    if lo is not None:
        ax.axvline(lo, color="red", ls="--", lw=1.5, label=f"min={lo:.1f}")
    if hi is not None:
        ax.axvline(hi, color="red", ls="--", lw=1.5, label=f"max={hi:.1f}")
    ax.set_title(label)
    if lo is not None or hi is not None:
        ax.legend(fontsize=8)
axes[1, 2].axis("off")

plt.suptitle("QC Histograms with Suggested Thresholds", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Suggested thresholds:")
print(f"  min_counts = {min_counts:.1f}")
print(f"  max_counts = {max_counts:.1f}")
print(f"  min_genes  = {min_genes:.1f}")
print(f"  max_mt     = {max_mt:.1f}%")

## Section 4: Quality Control — Part B: Spatial QC

Spatial QC detects artifacts invisible to global metrics: tissue folds, air bubbles,
edge effects. We use spatial neighbor-based z-scores to flag local outliers.

In [ ]:
%%time
# Spatial QC heatmaps
spatial_qc_plots(
    adata_8um,
    metrics=["total_counts", "n_genes_by_counts", "pct_counts_mt", "pct_counts_ribo", "complexity"],
    figsize=(25, 4), spot_size=0.3,
    save=str(FIG_DIR / "crc_spatial_qc_heatmaps.png"),
)

In [ ]:
%%time
# Spatial neighbors + outlier detection
print("Computing spatial neighbors (k=20)...")
sq.gr.spatial_neighbors(adata_8um, n_neighs=20, coord_type="generic")

outlier_metrics = ["total_counts", "pct_counts_mt", "n_genes_by_counts"]
total_outliers = np.zeros(adata_8um.shape[0], dtype=bool)
for metric in outlier_metrics:
    outliers = spatial_outlier_detection(adata_8um, metric=metric, z_threshold=3.0)
    n_out = outliers.sum()
    print(f"  {metric:25s}: {n_out:,} outliers ({100*n_out/len(outliers):.2f}%)")
    total_outliers = total_outliers | outliers.values

adata_8um.obs["spatial_outlier"] = total_outliers
print(f"\nTotal spatial outliers: {total_outliers.sum():,} ({100*total_outliers.mean():.2f}%)")

# Plot
fig, ax = plt.subplots(figsize=(8, 7))
coords = adata_8um.obsm["spatial"]
ax.scatter(coords[~total_outliers, 0], coords[~total_outliers, 1],
           s=0.1, c="lightgray", alpha=0.3, rasterized=True, label="Normal")
ax.scatter(coords[total_outliers, 0], coords[total_outliers, 1],
           s=1, c="red", alpha=0.6, rasterized=True, label="Outlier")
ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
ax.legend(markerscale=10, fontsize=10)
ax.set_title(f"Spatial outliers (n={total_outliers.sum():,})")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_spatial_outliers.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 5: Filtering & Normalization

Apply data-driven QC thresholds and spatial outlier removal, then normalize,
select highly variable genes, and compute PCA / UMAP / Leiden clusters.

In [ ]:
%%time
# --- Apply QC filters ---
n_before = adata_8um.shape[0]

min_counts = max(adata_8um.obs["total_counts"].quantile(0.01), 10)
max_counts = adata_8um.obs["total_counts"].quantile(0.995)
min_genes = max(adata_8um.obs["n_genes_by_counts"].quantile(0.01), 5)
max_mt = min(adata_8um.obs["pct_counts_mt"].quantile(0.95), 25)

keep = (
    (adata_8um.obs["total_counts"] >= min_counts) &
    (adata_8um.obs["total_counts"] <= max_counts) &
    (adata_8um.obs["n_genes_by_counts"] >= min_genes) &
    (adata_8um.obs["pct_counts_mt"] <= max_mt) &
    (~adata_8um.obs["spatial_outlier"].astype(bool))
)
adata_8um = adata_8um[keep].copy()

min_cells = max(int(0.001 * adata_8um.shape[0]), 3)
sc.pp.filter_genes(adata_8um, min_cells=min_cells)

print(f"Filtering summary:")
print(f"  Bins:  {n_before:,} -> {adata_8um.shape[0]:,}  "
      f"(removed {n_before - adata_8um.shape[0]:,}, "
      f"{100*(n_before - adata_8um.shape[0])/n_before:.1f}%)")
print(f"  Genes: {adata_8um.shape[1]:,} (min_cells={min_cells})")

In [ ]:
%%time
# --- Load pre-annotated data (normalized + PCA + UMAP + Leiden) ---
adata_8um = sc.read_h5ad(PRECOMPUTED_DIR / "crc_p1_annotated_50k.h5ad")
print(f"Loaded annotated data: {adata_8um.shape}")
print(f"Leiden clusters: {adata_8um.obs['leiden'].nunique()}")


In [ ]:
# --- UMAP colored by QC metrics + Leiden clusters ---
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, metric, cmap in zip(axes[:3],
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    ["viridis", "viridis", "RdYlBu_r"]):
    sc_p = ax.scatter(adata_8um.obsm["X_umap"][:, 0], adata_8um.obsm["X_umap"][:, 1],
                      c=adata_8um.obs[metric], s=0.3, cmap=cmap,
                      edgecolors="none", rasterized=True)
    ax.set_title(metric); plt.colorbar(sc_p, ax=ax, shrink=0.7)

# Leiden
for cl in sorted(adata_8um.obs["leiden"].unique(), key=int):
    mask = adata_8um.obs["leiden"] == cl
    axes[3].scatter(adata_8um.obsm["X_umap"][mask, 0], adata_8um.obsm["X_umap"][mask, 1],
                    s=0.3, label=cl, alpha=0.5, rasterized=True)
axes[3].set_title("Leiden clusters")
axes[3].legend(markerscale=5, fontsize=7, ncol=2, loc="best")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_umap_qc_leiden.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 6: Ground-Truth Cell Type Annotations

The 10x Genomics [CRC publication](https://www.nature.com/articles/s41588-025-02193-3)
provides two levels of cell type annotations:

- **UnsupervisedL1** (10 broad categories): Tumor, Fibroblast, T cells, B cells,
  Myeloid, Endothelial, Smooth Muscle, Intestinal Epithelial, Neuronal, Unknown
- **DeconvolutionLabel1** (38 detailed types): Tumor II, CAF, Goblet, Macrophage,
  CD4/CD8 T cells, Plasma, etc. — available for ~61% of bins (singlets only).

We visualize these ground-truth labels on UMAP and spatial coordinates.

In [ ]:
# --- Visualize ground-truth UnsupervisedL1 ---
ct_col = "UnsupervisedL1"
has_label = adata_8um.obs[ct_col].notna()
adata_labeled = adata_8um[has_label].copy()
print(f"Bins with {ct_col}: {has_label.sum():,} / {adata_8um.shape[0]:,}")

cell_types = sorted(adata_labeled.obs[ct_col].dropna().unique())
palette = dict(zip(cell_types, plt.cm.tab20(np.linspace(0, 1, len(cell_types)))))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ct in cell_types:
    mask = (adata_labeled.obs[ct_col] == ct).values
    axes[0].scatter(adata_labeled.obsm["X_umap"][mask, 0], adata_labeled.obsm["X_umap"][mask, 1],
                    s=0.5, label=ct, alpha=0.5, rasterized=True, color=palette[ct])
axes[0].legend(markerscale=8, fontsize=8, loc="best", framealpha=0.8)
axes[0].set_title(f"UMAP: {ct_col} (ground truth)"); axes[0].set_xlabel("UMAP 1"); axes[0].set_ylabel("UMAP 2")

coords = adata_labeled.obsm["spatial"]
for ct in cell_types:
    mask = (adata_labeled.obs[ct_col] == ct).values
    axes[1].scatter(coords[mask, 0], coords[mask, 1],
                    s=0.3, label=ct, alpha=0.4, rasterized=True, color=palette[ct])
axes[1].legend(markerscale=10, fontsize=8, loc="best", framealpha=0.8)
axes[1].set_title(f"Spatial: {ct_col}"); axes[1].set_aspect("equal"); axes[1].invert_yaxis(); axes[1].axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "crc_ground_truth_unsupervised.png", dpi=150, bbox_inches="tight")
plt.show()

# Proportions
fig, ax = plt.subplots(figsize=(10, 5))
props = adata_labeled.obs[ct_col].value_counts(normalize=True).sort_values(ascending=True)
colors = [palette.get(ct, "gray") for ct in props.index]
ax.barh(range(len(props)), props.values, color=colors)
ax.set_yticks(range(len(props))); ax.set_yticklabels(props.index)
ax.set_xlabel("Proportion"); ax.set_title(f"{ct_col} proportions")
for i, v in enumerate(props.values):
    ax.text(v + 0.005, i, f"{v:.1%}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_ground_truth_proportions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Detailed DeconvolutionLabel1 (singlets only) ---
deconv_col = "DeconvolutionLabel1"
has_deconv = adata_8um.obs[deconv_col].notna()
print(f"Bins with {deconv_col}: {has_deconv.sum():,} / {adata_8um.shape[0]:,}")

if has_deconv.sum() > 0:
    adata_deconv = adata_8um[has_deconv].copy()
    top_types = adata_deconv.obs[deconv_col].value_counts().head(12).index.tolist()
    adata_top = adata_deconv[adata_deconv.obs[deconv_col].isin(top_types)].copy()

    palette_d = dict(zip(top_types, plt.cm.tab20(np.linspace(0, 1, len(top_types)))))

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    for ct in top_types:
        mask = (adata_top.obs[deconv_col] == ct).values
        axes[0].scatter(adata_top.obsm["X_umap"][mask, 0], adata_top.obsm["X_umap"][mask, 1],
                        s=0.5, label=ct, alpha=0.5, rasterized=True, color=palette_d[ct])
    axes[0].legend(markerscale=8, fontsize=7, loc="best"); axes[0].set_title(f"UMAP: {deconv_col} (top 12)")

    coords = adata_top.obsm["spatial"]
    for ct in top_types:
        mask = (adata_top.obs[deconv_col] == ct).values
        axes[1].scatter(coords[mask, 0], coords[mask, 1],
                        s=0.3, label=ct, alpha=0.4, rasterized=True, color=palette_d[ct])
    axes[1].legend(markerscale=10, fontsize=7, loc="best")
    axes[1].set_title(f"Spatial: {deconv_col}"); axes[1].set_aspect("equal"); axes[1].invert_yaxis(); axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_ground_truth_deconvolution.png", dpi=150, bbox_inches="tight")
    plt.show()

## Section 7: Marker-Based Annotation & Comparison to Ground Truth

We perform our own marker-based annotation using canonical CRC marker gene sets,
then compare against the published ground-truth labels.

In [ ]:
%%time
# --- Marker-based annotation for CRC ---
print("Scoring marker panels...")

marker_sets = {
    "Tumor_Epithelial": ["EPCAM", "KRT8", "KRT18", "KRT19", "KRT20", "CEACAM5", "MUC1"],
    "Fibroblast":       ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "COL6A1"],
    "Endothelial":      ["PECAM1", "VWF", "EMCN", "KDR", "RAMP2", "PLVAP"],
    "Myeloid":          ["LST1", "TYROBP", "FCER1G", "CTSS", "AIF1", "C1QC", "C1QB"],
    "T_NK":             ["CD3D", "CD3E", "TRBC1", "TRBC2", "NKG7", "IL7R", "LTB"],
    "B_Plasma":         ["MS4A1", "CD79A", "CD79B", "MZB1", "JCHAIN", "IGKC", "CD74"],
    "Smooth_Muscle":    ["ACTA2", "MYH11", "DES", "TAGLN", "CNN1", "ACTG2"],
    "Goblet":           ["MUC2", "TFF3", "FCGBP", "CLCA1", "ZG16", "AGR2"],
    "Enterocyte":       ["FABP1", "FABP2", "ALPI", "VIL1", "SIS"],
    "Mast":             ["TPSAB1", "TPSB2", "KIT", "CPA3", "HDC"],
}

present_markers = {}
for ct, genes in marker_sets.items():
    present = [g for g in genes if g in adata_8um.var_names]
    present_markers[ct] = present
    print(f"  {ct:20s}  {len(present):>2}/{len(genes)} genes  {present}")

score_cols = []
for ct, genes in present_markers.items():
    if len(genes) >= 2:
        score_col = f"{ct}_score"
        sc.tl.score_genes(adata_8um, gene_list=genes, score_name=score_col, use_raw=False)
        score_cols.append(score_col)
    else:
        print(f"  WARNING: {ct} has <2 markers — skipping")

score_df = adata_8um.obs[score_cols].copy()
adata_8um.obs["cell_type_marker"] = score_df.idxmax(axis=1).str.replace("_score", "", regex=False)

max_score = score_df.max(axis=1)
sorted_scores = np.sort(score_df.values, axis=1)
second_score = sorted_scores[:, -2] if sorted_scores.shape[1] > 1 else np.zeros(len(score_df))
margin = max_score.values - second_score
adata_8um.obs.loc[(max_score < 0.05) | (margin < 0.02), "cell_type_marker"] = "Low_confidence"

print("\nMarker-based cell type distribution:")
print(adata_8um.obs["cell_type_marker"].value_counts().to_string())


In [ ]:
# --- Marker score heatmap ---
score_cols = [c for c in adata_8um.obs.columns if c.endswith("_score")]
score_summary = (
    adata_8um.obs.groupby("cell_type_marker")[score_cols]
    .median()
    .rename(columns=lambda x: x.replace("_score", ""))
)
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(score_summary.values, aspect="auto", cmap="viridis")
ax.set_xticks(np.arange(score_summary.shape[1]))
ax.set_xticklabels(score_summary.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(score_summary.shape[0]))
ax.set_yticklabels(score_summary.index)
ax.set_title("Median marker scores by assigned cell type")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_marker_score_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Compare marker-based vs ground-truth (UnsupervisedL1) ---
marker_to_gt = {
    "Tumor_Epithelial": "Tumor",
    "Fibroblast":       "Fibroblast",
    "Endothelial":      "Endothelial",
    "Myeloid":          "Myeloid",
    "T_NK":             "T cells",
    "B_Plasma":         "B cells",
    "Smooth_Muscle":    "Smooth Muscle",
    "Goblet":           "Intestinal Epithelial",
    "Enterocyte":       "Intestinal Epithelial",
    "Mast":             "Myeloid",
}

adata_8um.obs["marker_mapped"] = adata_8um.obs["cell_type_marker"].map(marker_to_gt).fillna("Other")

has_both = (adata_8um.obs["UnsupervisedL1"].notna()) & (adata_8um.obs["cell_type_marker"] != "Low_confidence")
if has_both.sum() > 0:
    agree = (adata_8um.obs.loc[has_both, "marker_mapped"] == adata_8um.obs.loc[has_both, "UnsupervisedL1"])
    print(f"Agreement (marker vs ground-truth): {agree.mean():.1%}  ({has_both.sum():,} bins)")

    cm = pd.crosstab(
        adata_8um.obs.loc[has_both, "marker_mapped"],
        adata_8um.obs.loc[has_both, "UnsupervisedL1"],
        normalize="index",
    )
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", ax=ax)
    ax.set_xlabel("Ground Truth (UnsupervisedL1)")
    ax.set_ylabel("Marker-Based (mapped)")
    ax.set_title(f"Annotation Agreement: {agree.mean():.1%}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_annotation_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# --- Spatial map of marker-based annotations ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ct_col = "cell_type_marker"
cell_types = sorted(adata_8um.obs[ct_col].unique())
palette = dict(zip(cell_types, plt.cm.tab20(np.linspace(0, 1, len(cell_types)))))

for ct in cell_types:
    mask = (adata_8um.obs[ct_col] == ct).values
    axes[0].scatter(adata_8um.obsm["X_umap"][mask, 0], adata_8um.obsm["X_umap"][mask, 1],
                    s=0.5, label=ct, alpha=0.5, rasterized=True, color=palette[ct])
axes[0].legend(markerscale=8, fontsize=7, loc="best"); axes[0].set_title("UMAP: Marker-based")

coords = adata_8um.obsm["spatial"]
for ct in cell_types:
    mask = (adata_8um.obs[ct_col] == ct).values
    axes[1].scatter(coords[mask, 0], coords[mask, 1],
                    s=0.3, label=ct, alpha=0.4, rasterized=True, color=palette[ct])
axes[1].legend(markerscale=10, fontsize=7, loc="best")
axes[1].set_title("Spatial: Marker-based"); axes[1].set_aspect("equal"); axes[1].invert_yaxis(); axes[1].axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "crc_marker_cell_type_maps.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 8: Neighborhood Analysis

Which cell types spatially co-localize? In CRC we expect:
- **Tumor** self-clusters in tumor nests
- **Fibroblasts (CAFs)** surround tumor regions
- **Immune cells** concentrate in the tumor periphery and stroma
- **Intestinal epithelium** (goblet, enterocytes) in normal tissue regions

In [ ]:
%%time
# --- Neighborhood enrichment using ground-truth labels ---
ct_key = "UnsupervisedL1"
adata_nhood = adata_8um[adata_8um.obs[ct_key].notna()].copy()
adata_nhood.obs[ct_key] = adata_nhood.obs[ct_key].astype("category")

sq.gr.spatial_neighbors(adata_nhood, n_neighs=20, coord_type="generic")
sq.gr.nhood_enrichment(adata_nhood, cluster_key=ct_key)

fig, ax = plt.subplots(figsize=(8, 7))
sq.pl.nhood_enrichment(adata_nhood, cluster_key=ct_key, ax=ax)
ax.set_title("Neighborhood Enrichment (UnsupervisedL1)")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_nhood_enrichment.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
%%time
# --- Co-occurrence analysis ---
try:
    sq.gr.co_occurrence(adata_nhood, cluster_key=ct_key)
    sq.pl.co_occurrence(adata_nhood, cluster_key=ct_key)
    plt.savefig(FIG_DIR / "crc_co_occurrence.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"Co-occurrence analysis: {e}")
    print("This can be slow/memory-intensive on large datasets. Skipping.")

del adata_nhood; gc.collect()

## Section 9: Spatially Variable Genes (SVGs)

Identify genes with significant spatial structure using Moran's I autocorrelation.
We subsample to 10K bins for computational efficiency.

In [ ]:
%%time
# --- SVG detection (subsampled) ---
n_svg = min(10_000, adata_8um.shape[0])
print(f"Subsampling to {n_svg:,} bins for SVG analysis...")
adata_svg = sc.pp.subsample(adata_8um, n_obs=n_svg, copy=True)
sq.gr.spatial_neighbors(adata_svg, n_neighs=20, coord_type="generic")

if "highly_variable" in adata_svg.var.columns:
    adata_svg_hvg = adata_svg[:, adata_svg.var["highly_variable"]].copy()
else:
    adata_svg_hvg = adata_svg.copy()

print(f"Running Moran's I on {adata_svg_hvg.shape[1]} genes...")
sq.gr.spatial_autocorr(adata_svg_hvg, mode="moran", n_jobs=1)

morans = adata_svg_hvg.uns["moranI"].sort_values("I", ascending=False)
print(f"\nTop 20 SVGs:")
print(morans.head(20)[["I", "pval_norm"]].to_string())

In [ ]:
# --- Spatial maps of top SVGs ---
top_svgs = morans.head(8).index.tolist()
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
coords = adata_svg.obsm["spatial"]

for ax, gene in zip(axes.flat, top_svgs):
    if gene in adata_svg.var_names:
        vals = adata_svg[:, gene].X
        expr = vals.toarray().flatten() if sparse.issparse(vals) else np.asarray(vals).flatten()
    else:
        expr = np.zeros(adata_svg.shape[0])
    sc_p = ax.scatter(coords[:, 0], coords[:, 1], c=expr, s=1, cmap="magma",
                      edgecolors="none", rasterized=True)
    ax.set_title(f"{gene}\n(I={morans.loc[gene, 'I']:.3f})", fontsize=10)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    plt.colorbar(sc_p, ax=ax, shrink=0.6)

plt.suptitle("Top Spatially Variable Genes", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_svgs.png", dpi=150, bbox_inches="tight")
plt.show()

del adata_svg, adata_svg_hvg; gc.collect()

## Section 10: Cell-Cell Communication

Identify ligand-receptor interactions between spatially proximal cell types.

In [ ]:
%%time
# --- L-R analysis (subsampled) ---
ct_key = "UnsupervisedL1"
adata_lr_src = adata_8um[adata_8um.obs[ct_key].notna()].copy()
n_lr = min(20_000, adata_lr_src.shape[0])
adata_lr = sc.pp.subsample(adata_lr_src, n_obs=n_lr, copy=True)
del adata_lr_src
adata_lr.obs[ct_key] = adata_lr.obs[ct_key].astype("category")

sq.gr.spatial_neighbors(adata_lr, n_neighs=20, coord_type="generic")

print(f"Running L-R analysis on {adata_lr.shape[0]:,} bins, 100 permutations...")
try:
    sq.gr.ligrec(adata_lr, cluster_key=ct_key, n_perms=100, use_raw=False)
except TypeError:
    # Older squidpy versions may need different params
    sq.gr.ligrec(adata_lr, cluster_key=ct_key, n_perms=100)
print("Done.")

In [ ]:
# --- Plot significant L-R pairs ---
try:
    sq.pl.ligrec(adata_lr, cluster_key=ct_key,
                 pvalue_threshold=0.05, mean_range=(0.3, np.inf))
    plt.savefig(FIG_DIR / "crc_ligrec.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"L-R plot: {e}")
    print("Try adjusting pvalue_threshold or mean_range if no pairs pass filters.")

del adata_lr; gc.collect()

## Section 11: Marker Validation

Validate annotations using a dotplot of canonical markers and visualize
the **Periphery** labels (Tumor / Tissue / 50 µm border).

In [ ]:
# --- Dotplot of canonical markers by ground-truth cell type ---
validation = {
    "Tumor":     ["EPCAM", "KRT20", "KRT8"],
    "Fibroblast": ["COL1A1", "DCN", "LUM"],
    "T cells":   ["CD3D", "CD3E"],
    "B cells":   ["MS4A1", "CD79A"],
    "Myeloid":   ["TYROBP", "C1QC", "AIF1"],
    "Endothelial": ["PECAM1", "VWF"],
    "Smooth Muscle": ["ACTA2", "MYH11"],
    "Intestinal Epithelial": ["MUC2", "TFF3", "FABP1"],
}

valid = {}
for ct, genes in validation.items():
    present = [g for g in genes if g in adata_8um.var_names]
    if present:
        valid[ct] = present

adata_gt = adata_8um[adata_8um.obs["UnsupervisedL1"].notna()].copy()
if valid:
    try:
        sc.pl.dotplot(adata_gt, var_names=valid, groupby="UnsupervisedL1",
                      use_raw=False, standard_scale="var",
                      title="Canonical markers by ground-truth cell type")
        plt.savefig(FIG_DIR / "crc_marker_dotplot.png", dpi=150, bbox_inches="tight")
        plt.show()
    except Exception as e:
        print(f"Dotplot: {e}")
        flat = [g for gs in valid.values() for g in gs]
        sc.pl.dotplot(adata_gt, var_names=flat, groupby="UnsupervisedL1", use_raw=False)
        plt.show()

In [ ]:
# --- Periphery labels: Tumor vs Tissue vs 50µm border ---
if "Periphery" in adata_8um.obs.columns and adata_8um.obs["Periphery"].notna().any():
    fig, ax = plt.subplots(figsize=(9, 8))
    coords = adata_8um.obsm["spatial"]
    periph_colors = {"Tumor": "red", "Tissue": "steelblue", "50 micron": "gold"}

    for label, color in periph_colors.items():
        mask = (adata_8um.obs["Periphery"] == label).values
        if mask.any():
            ax.scatter(coords[mask, 0], coords[mask, 1], s=0.3, c=color,
                       label=f"{label} ({mask.sum():,})", alpha=0.4, rasterized=True)

    ax.legend(markerscale=10, fontsize=10)
    ax.set_title("Tumor Periphery Classification")
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_periphery.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Periphery labels not available.")

## Section 12: Summary & Best Practices

### What we demonstrated
1. **Real QC filtering** on a clinically relevant human CRC dataset — removed bins with
   low UMIs, high mitochondrial content, and spatial outliers.
2. **Ground-truth cell type annotations** from a Nature Genetics 2025 publication —
   10 broad categories (UnsupervisedL1) and 38 detailed deconvolution labels.
3. **Marker-based annotation** compared against ground truth — quantified agreement.
4. **Spatial analysis** — neighborhood enrichment reveals tumor-stroma-immune organization,
   SVGs show tissue zonation, L-R interactions identify tumor-immune signaling.
5. **Tumor periphery analysis** — the 50 µm boundary annotation is unique to this dataset.

### Key takeaways
- **Always validate annotations** — comparing marker-based scoring against published
  ground truth shows where simple approaches succeed and where they need refinement.
- **Spatial QC catches what global QC misses** — tissue folds, edge artifacts.
- **Data-driven thresholds** with biological guardrails are more robust than arbitrary cutoffs.
- **Published datasets with annotations** are invaluable for benchmarking methods.

### Caveats
- This is FFPE tissue — RNA fragmentation introduces 3' bias.
- 8 µm bins average 1–3 cells. For single-cell resolution, use segmented outputs or
  deconvolution (as done in the original publication).
- The 250K bin subsample captures the full tissue structure but excludes some bins.

### References
- de Oliveira et al. *High-definition spatial transcriptomic profiling of immune cell
  populations in colorectal cancer.* Nature Genetics (2025).
- [GitHub: 10XGenomics/HumanColonCancer_VisiumHD](https://github.com/10XGenomics/HumanColonCancer_VisiumHD)
- [Dataset page](https://www.10xgenomics.com/datasets/visium-hd-cytassist-gene-expression-libraries-of-human-crc)